![ngo_project_image](ngo_project_image.jpg)

GoodThought NGO has been a catalyst for positive change, focusing its efforts on education, healthcare, and sustainable development to make a significant difference in communities worldwide. With this mission, GoodThought has orchestrated an array of assignments aimed at uplifting underprivileged populations and fostering long-term growth.

This project offers a hands-on opportunity to explore how data-driven insights can direct and enhance these humanitarian efforts. In this project, you'll engage with the GoodThought PostgreSQL database, which encapsulates detailed records of assignments, funding, impacts, and donor activities from 2010 to 2023. This comprehensive dataset includes:

- **`Assignments`:** Details about each project, including its name, duration (start and end dates), budget, geographical region, and the impact score.
- **`Donations`:** Records of financial contributions, linked to specific donors and assignments, highlighting how financial support is allocated and utilized.
- **`Donors`:** Information on individuals and organizations that fund GoodThought's projects, including donor types.

Refer to the below ERD diagram for a visual representation of the relationships between these data tables:
<img src="erd.png" alt="ERD" width="50%" height="50%">

You will execute SQL queries to answer two questions, as listed in the instructions. Good luck!

---
# Local setup with DuckDB

No Docker, no PostgreSQL, no pgAdmin. DuckDB reads your CSV files directly and runs the same SQL.

**Put these three files in the same folder as this notebook** (next to `erd.png` and `ngo_project_image.jpg`):

- `assignments.csv`
- `donors.csv`
- `donations.csv`

In [4]:

# Run once. Remove the # if you need to install.

%pip install duckdb pandas


Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path

needed = ["assignments.csv", "donors.csv", "donations.csv"]
missing = [f for f in needed if not Path(f).exists()]

if missing:
    print("Missing files:", missing)
    print("Put them in this folder:", Path.cwd())
else:
    print("All 3 CSV files found.")

All 3 CSV files found.


## 1. Load the CSVs into DuckDB

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect()  # in-memory database, nothing to install or start

# Each CSV has an extra "index" column from the DataCamp export, so we drop it.
for t in ["assignments", "donors", "donations"]:
    con.sql(f'''
        CREATE OR REPLACE TABLE {t} AS
        SELECT * EXCLUDE ("index")
        FROM read_csv_auto('{t}.csv', header=true)
    ''')

con.sql("SHOW TABLES").show()

┌─────────────┐
│    name     │
│   varchar   │
├─────────────┤
│ assignments │
│ donations   │
│ donors      │
└─────────────┘



## 2. Check the data loaded (counts should match DataCamp)

In [3]:
con.sql('''
    SELECT 'assignments' AS table_name, COUNT(*) AS row_count FROM assignments
    UNION ALL SELECT 'donors', COUNT(*) FROM donors
    UNION ALL SELECT 'donations', COUNT(*) FROM donations
''').show()

con.sql("DESCRIBE assignments").show()
con.sql("DESCRIBE donors").show()
con.sql("DESCRIBE donations").show()

┌─────────────┬───────────┐
│ table_name  │ row_count │
│   varchar   │   int64   │
├─────────────┼───────────┤
│ assignments │      5000 │
│ donors      │      5000 │
│ donations   │      5000 │
└─────────────┴───────────┘

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ assignment_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ assignment_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_date      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ end_date        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ budget          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ region          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ impact_score    │ DOUBLE      │ Y

## 3. Task 1: `highest_donation_assignments`

Top 5 assignments by total donations, with region, rounded total and donor type.

In [4]:
highest_donation_assignments = con.sql('''
    WITH donation_details AS (
        SELECT
            d.assignment_id,
            ROUND(SUM(d.amount), 2) AS total_sum_donations,
            dn.donor_type
        FROM donations d
        JOIN donors dn ON d.donor_id = dn.donor_id
        GROUP BY d.assignment_id, dn.donor_type
    )
    SELECT
        a.assignment_name,
        a.region,
        dd.total_sum_donations AS rounded_total_donation_amount,
        dd.donor_type
    FROM assignments a
    JOIN donation_details dd ON a.assignment_id = dd.assignment_id
    ORDER BY dd.total_sum_donations DESC
    LIMIT 5
''').df()

highest_donation_assignments

,assignment_name,region,rounded_total_donation_amount,donor_type
0,Assignment_3033,East,3840.66,Individual
1,Assignment_300,West,3133.98,Organization
2,Assignment_4114,North,2778.57,Organization
3,Assignment_1765,West,2626.98,Organization
4,Assignment_268,East,2488.69,Individual


## 4. Task 2: `top_regional_impact_assignments`

Highest-impact assignment in each region, among assignments that received at least one donation.
`ROW_NUMBER()` with a tiebreaker guarantees exactly one row per region.

In [5]:
top_regional_impact_assignments = con.sql('''
    WITH donation_counts AS (
        SELECT assignment_id, COUNT(donation_id) AS total_donations
        FROM donations
        GROUP BY assignment_id
    ),
    ranked_assignments AS (
        SELECT
            a.assignment_name,
            a.region,
            a.impact_score,
            dc.total_donations,
            ROW_NUMBER() OVER (
                PARTITION BY a.region
                ORDER BY a.impact_score DESC, a.assignment_name ASC
            ) AS rank_in_region
        FROM assignments a
        JOIN donation_counts dc ON a.assignment_id = dc.assignment_id
    )
    SELECT assignment_name, region, impact_score, total_donations
    FROM ranked_assignments
    WHERE rank_in_region = 1
    ORDER BY region
''').df()

top_regional_impact_assignments

,assignment_name,region,impact_score,total_donations
0,Assignment_316,East,10.00,2
1,Assignment_2253,North,9.99,1
2,Assignment_3547,South,10.00,1
3,Assignment_2794,West,9.99,2


## 5. Sanity checks

In [6]:
print("Task 1 rows (expect 5):", len(highest_donation_assignments))
print("Task 2 rows vs distinct regions (must be equal):",
      len(top_regional_impact_assignments),
      top_regional_impact_assignments["region"].nunique())

Task 1 rows (expect 5): 5
Task 2 rows vs distinct regions (must be equal): 4 4


## 6. Your interpretation (write this yourself)

In 2 or 3 sentences, answer:
- Are the top-funded assignments also the highest-impact ones?
- Which donor type is behind the biggest donations, and what should the NGO do with that?
- How would you explain the business value of this analysis to a hiring manager?

In [10]:
con.sql("SELECT * FROM assignments LIMIT 10").df()


,assignment_id,assignment_name,start_date,end_date,budget,region,impact_score
0,1,Assignment_1,2021-10-17,2021-12-04,-32322.03,West,5.55
1,2,Assignment_2,2020-10-26,2020-11-28,57278.40,South,1.45
2,3,Assignment_3,2021-08-11,2022-03-17,40414.51,West,2.34
3,4,Assignment_4,2021-11-22,2022-05-17,31732.48,East,7.05
4,5,Assignment_5,2020-11-22,2021-07-10,13548.22,North,5.29
5,6,Assignment_6,2020-01-02,2021-01-17,46884.12,East,6.38
6,7,Assignment_7,2020-07-20,2020-11-18,15638.08,East,5.12
7,8,Assignment_8,2021-02-01,2021-03-10,83765.95,North,9.33
8,9,Assignment_9,2021-12-01,2021-07-12,94549.41,North,8.43
9,10,Assignment_10,2020-10-11,2020-12-23,7516.23,South,9.82


In [11]:
# A. Impact score of the 5 top-funded assignments
con.sql('''
    SELECT a.assignment_name, a.region, a.impact_score,
           ROUND(SUM(d.amount), 2) AS total_donations
    FROM donations d
    JOIN assignments a ON a.assignment_id = d.assignment_id
    GROUP BY a.assignment_name, a.region, a.impact_score
    ORDER BY total_donations DESC
    LIMIT 5
''').df()

,assignment_name,region,impact_score,total_donations
0,Assignment_3033,East,7.38,3840.66
1,Assignment_4114,North,4.78,3647.86
2,Assignment_2406,East,2.90,3514.37
3,Assignment_2225,East,4.50,3471.53
4,Assignment_1740,West,4.52,3419.55


In [12]:
# C. Which donor type gives the most overall?
con.sql('''
    SELECT dn.donor_type,
           COUNT(*) AS num_donations,
           ROUND(SUM(d.amount), 2) AS total_given,
           ROUND(AVG(d.amount), 2) AS avg_donation
    FROM donations d
    JOIN donors dn ON dn.donor_id = d.donor_id
    GROUP BY dn.donor_type
    ORDER BY total_given DESC
''').df()

,donor_type,num_donations,total_given,avg_donation
0,Organization,1693,860175.04,508.08
1,Individual,1680,840073.79,500.04
2,Corporate,1627,826330.46,507.89


In [13]:
# B. Does more money go with higher impact? (-1 to 1; near 0 = no link)
con.sql('''
    WITH t AS (
        SELECT a.assignment_id, a.impact_score, SUM(d.amount) AS total
        FROM assignments a
        JOIN donations d ON d.assignment_id = a.assignment_id
        GROUP BY a.assignment_id, a.impact_score
    )
    SELECT ROUND(corr(total, impact_score), 3) AS correlation FROM t
''').df()

,correlation
0,-0.018
